# D3 站点坐标修正（插值计算）

## 特点
- 只处理 ML、HV、FF 类型站点
- 使用相邻里程桩**插值**计算精确坐标
- 绘制 Caltrans 里程桩参考点

In [1]:
import pandas as pd
import numpy as np
import json
import os
import glob
import folium
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm

# ============== 配置 ==============

# Caltrans GeoJSON 文件路径
CALTRANS_GEOJSON = "./SHN_Postmiles_Tenth.geojson"

# PeMS 元数据目录
META_DIR = "../d03_meta"

# 只处理这些站点类型
TARGET_TYPES = ['ML', 'HV', 'FF']

# 输出目录
OUTPUT_DIR = "./output/d3_interpolation_correction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")
print(f"目标站点类型: {TARGET_TYPES}")

配置完成！
目标站点类型: ['ML', 'HV', 'FF']


## 1. 加载 PeMS D3 元数据

In [2]:
META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

# 加载 D3 元数据
pattern = os.path.join(META_DIR, "d03_text_meta_*.txt")
files = glob.glob(pattern)
if not files:
    raise FileNotFoundError("未找到 D3 元数据文件")

meta_file = sorted(files)[-1]
print(f"使用元数据文件: {meta_file}")

pems_df = pd.read_csv(meta_file, sep='\t', names=META_COLUMNS, header=0, 
                      dtype={'ID': str, 'Fwy': str})
print(f"总记录数: {len(pems_df)}")

使用元数据文件: ../d03_meta/d03_text_meta_2025_12_30.txt
总记录数: 1903


In [3]:
# 筛选目标站点类型
pems_target = pems_df[
    pems_df['Type'].isin(TARGET_TYPES) &
    pems_df['Latitude'].notna() & 
    pems_df['Longitude'].notna() &
    pems_df['Abs_PM'].notna() &
    pems_df['Fwy'].notna() &
    (pems_df['Latitude'] > 30) &
    (pems_df['Latitude'] < 42)
].copy()

print(f"目标站点数: {len(pems_target)}")
print(f"\n类型分布:")
print(pems_target['Type'].value_counts())
print(f"\n高速分布:")
print(pems_target['Fwy'].value_counts())

目标站点数: 1192

类型分布:
Type
ML    883
HV    281
FF     28
Name: count, dtype: int64

高速分布:
Fwy
50     321
80     297
99     218
5      151
51      48
65      34
20      22
89      18
12      16
70      15
113      9
160      6
28       6
49       6
45       6
162      6
244      5
267      4
505      2
275      1
16       1
Name: count, dtype: int64


In [4]:
# 获取需要加载的高速列表
target_routes_int = []
for r in pems_target['Fwy'].unique():
    try:
        target_routes_int.append(int(r))
    except:
        pass

print(f"需要加载的高速: {sorted(target_routes_int)}")

需要加载的高速: [5, 12, 16, 20, 28, 45, 49, 50, 51, 65, 70, 80, 89, 99, 113, 160, 162, 244, 267, 275, 505]


## 2. 加载 Caltrans 官方数据

In [5]:
print(f"加载 Caltrans 数据: {CALTRANS_GEOJSON}")
print("文件较大，请稍候...")

with open(CALTRANS_GEOJSON, 'r') as f:
    data = json.load(f)

print(f"总特征数: {len(data['features'])}")

加载 Caltrans 数据: ./SHN_Postmiles_Tenth.geojson
文件较大，请稍候...
总特征数: 311198


In [6]:
# 提取目标高速数据
records = []
for feature in tqdm(data['features'], desc="筛选 Caltrans 数据"):
    props = feature.get('properties', {})
    route = props.get('Route')
    
    if route not in target_routes_int:
        continue
    
    geom = feature.get('geometry', {})
    coords = geom.get('coordinates', [None, None])
    
    odometer = props.get('Odometer')
    if odometer is None or coords[0] is None:
        continue
    
    records.append({
        'Route': route,
        'County': props.get('County'),
        'PM': props.get('PM'),
        'Odometer': odometer,
        'AlignCode': props.get('AlignCode'),
        'Direction': props.get('Direction'),
        'Longitude': coords[0],
        'Latitude': coords[1],
    })

caltrans_df = pd.DataFrame(records)
print(f"\nCaltrans 有效里程桩数: {len(caltrans_df)}")

筛选 Caltrans 数据: 100%|██████████| 311198/311198 [00:00<00:00, 905837.93it/s]



Caltrans 有效里程桩数: 64124


## 3. 插值修正坐标

In [7]:
def calc_distance_feet(lat1, lon1, lat2, lon2):
    """计算两点距离（英尺）"""
    R = 3958.8 * 5280
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def interpolate_coordinate(abs_pm, pm1, lat1, lon1, pm2, lat2, lon2):
    """
    在两个里程桩之间线性插值计算坐标
    
    参数:
        abs_pm: 目标 Abs_PM 值
        pm1, lat1, lon1: 前一个里程桩
        pm2, lat2, lon2: 后一个里程桩
    
    返回:
        (lat, lon): 插值后的坐标
    """
    if pm2 == pm1:
        return lat1, lon1
    
    # 计算插值比例
    t = (abs_pm - pm1) / (pm2 - pm1)
    
    # 线性插值
    lat = lat1 + t * (lat2 - lat1)
    lon = lon1 + t * (lon2 - lon1)
    
    return lat, lon


def correct_coordinates_interpolation(pems_df, caltrans_df):
    """
    使用插值方法修正坐标
    
    对于每个 PeMS 站点:
    1. 找到其 Abs_PM 前后的两个 Caltrans 里程桩
    2. 在两点之间线性插值计算精确坐标
    """
    results = []
    
    # 预先按 Route 和 AlignCode 分组并排序
    caltrans_grouped = {}
    for route in caltrans_df['Route'].unique():
        route_data = caltrans_df[caltrans_df['Route'] == route]
        
        # Right 方向按 Odometer 排序
        right = route_data[route_data['AlignCode'].isin(['Right', 'Right Side'])].sort_values('Odometer')
        # Left 方向按 Odometer 排序
        left = route_data[route_data['AlignCode'].isin(['Left', 'Left Side'])].sort_values('Odometer')
        
        caltrans_grouped[route] = {
            'right': right.reset_index(drop=True),
            'left': left.reset_index(drop=True),
        }
    
    for _, row in tqdm(pems_df.iterrows(), total=len(pems_df), desc="插值修正"):
        result = row.to_dict()
        result['Original_Lat'] = row['Latitude']
        result['Original_Lon'] = row['Longitude']
        
        # 获取路线号
        try:
            route = int(row['Fwy'])
        except:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Status'] = 'invalid_fwy'
            results.append(result)
            continue
        
        if route not in caltrans_grouped:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Status'] = 'no_caltrans_route'
            results.append(result)
            continue
        
        # 方向映射
        if row['Dir'] in ['N', 'E']:
            caltrans_sorted = caltrans_grouped[route]['right']
        else:
            caltrans_sorted = caltrans_grouped[route]['left']
        
        if len(caltrans_sorted) == 0:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Status'] = 'no_caltrans_direction'
            results.append(result)
            continue
        
        abs_pm = row['Abs_PM']
        odometers = caltrans_sorted['Odometer'].values
        
        # 找到 abs_pm 在排序数组中的位置
        idx = np.searchsorted(odometers, abs_pm)
        
        # 边界情况处理
        if idx == 0:
            # abs_pm 在最小值之前，使用第一个点
            nearest = caltrans_sorted.iloc[0]
            result['Corrected_Lat'] = nearest['Latitude']
            result['Corrected_Lon'] = nearest['Longitude']
            result['Interp_Type'] = 'before_start'
            result['PM1'] = nearest['Odometer']
            result['PM2'] = None
        elif idx >= len(odometers):
            # abs_pm 在最大值之后，使用最后一个点
            nearest = caltrans_sorted.iloc[-1]
            result['Corrected_Lat'] = nearest['Latitude']
            result['Corrected_Lon'] = nearest['Longitude']
            result['Interp_Type'] = 'after_end'
            result['PM1'] = nearest['Odometer']
            result['PM2'] = None
        else:
            # 正常情况：在两个里程桩之间插值
            point1 = caltrans_sorted.iloc[idx - 1]
            point2 = caltrans_sorted.iloc[idx]
            
            lat, lon = interpolate_coordinate(
                abs_pm,
                point1['Odometer'], point1['Latitude'], point1['Longitude'],
                point2['Odometer'], point2['Latitude'], point2['Longitude']
            )
            
            result['Corrected_Lat'] = lat
            result['Corrected_Lon'] = lon
            result['Interp_Type'] = 'interpolated'
            result['PM1'] = point1['Odometer']
            result['PM2'] = point2['Odometer']
        
        # 计算修正距离
        correction_dist = calc_distance_feet(
            row['Latitude'], row['Longitude'],
            result['Corrected_Lat'], result['Corrected_Lon']
        )
        
        result['Correction_Dist_ft'] = correction_dist
        result['Status'] = 'corrected'
        
        results.append(result)
    
    return pd.DataFrame(results)


print("插值修正函数定义完成")

插值修正函数定义完成


In [8]:
# 执行插值修正
pems_corrected = correct_coordinates_interpolation(pems_target, caltrans_df)

print("\n修正状态统计:")
print(pems_corrected['Status'].value_counts())

print("\n插值类型统计:")
corrected = pems_corrected[pems_corrected['Status'] == 'corrected']
print(corrected['Interp_Type'].value_counts())

插值修正: 100%|██████████| 1192/1192 [00:00<00:00, 4045.88it/s]



修正状态统计:
Status
corrected    1192
Name: count, dtype: int64

插值类型统计:
Interp_Type
interpolated    1192
Name: count, dtype: int64


In [9]:
# 修正距离统计
print(f"成功修正站点数: {len(corrected)}")
print("\n修正距离统计 (英尺):")
print(f"  平均: {corrected['Correction_Dist_ft'].mean():.1f}")
print(f"  中位数: {corrected['Correction_Dist_ft'].median():.1f}")
print(f"  最小: {corrected['Correction_Dist_ft'].min():.1f}")
print(f"  最大: {corrected['Correction_Dist_ft'].max():.1f}")

print("\n修正距离分布:")
bins = [(0, 50), (50, 100), (100, 200), (200, 500), (500, 1000), (1000, float('inf'))]
for low, high in bins:
    count = ((corrected['Correction_Dist_ft'] >= low) & (corrected['Correction_Dist_ft'] < high)).sum()
    label = f"{low}-{high}" if high != float('inf') else f">{low}"
    pct = count / len(corrected) * 100
    print(f"  {label} ft: {count} 个站点 ({pct:.1f}%)")

成功修正站点数: 1192

修正距离统计 (英尺):
  平均: 894.8
  中位数: 109.1
  最小: 1.8
  最大: 27422.9

修正距离分布:
  0-50 ft: 353 个站点 (29.6%)
  50-100 ft: 198 个站点 (16.6%)
  100-200 ft: 160 个站点 (13.4%)
  200-500 ft: 201 个站点 (16.9%)
  500-1000 ft: 17 个站点 (1.4%)
  >1000 ft: 263 个站点 (22.1%)


In [10]:
# 按类型统计
print("各类型平均修正距离:")
type_stats = corrected.groupby('Type')['Correction_Dist_ft'].agg(['mean', 'median', 'max', 'count'])
type_stats.columns = ['平均(ft)', '中位数(ft)', '最大(ft)', '站点数']
print(type_stats.round(1))

各类型平均修正距离:
      平均(ft)  中位数(ft)   最大(ft)  站点数
Type                               
FF     643.3     86.7   6856.5   28
HV     274.9    101.9   1283.2  281
ML    1100.1    117.1  27422.9  883


In [11]:
# 保存结果
output_file = os.path.join(OUTPUT_DIR, 'pems_d3_interpolated.csv')
pems_corrected.to_csv(output_file, index=False)
print(f"已保存: {output_file}")

已保存: ./output/d3_interpolation_correction/pems_d3_interpolated.csv


## 4. 导出修正后的元数据

In [12]:
# 生成用于构图的元数据
pems_for_graph = pems_corrected[pems_corrected['Status'] == 'corrected'].copy()
pems_for_graph['Latitude'] = pems_for_graph['Corrected_Lat']
pems_for_graph['Longitude'] = pems_for_graph['Corrected_Lon']

graph_columns = ['ID', 'Fwy', 'Dir', 'District', 'County', 'City',
                 'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
                 'Type', 'Lanes', 'Name']

available_cols = [c for c in graph_columns if c in pems_for_graph.columns]
pems_export = pems_for_graph[available_cols].copy()

export_file = os.path.join(OUTPUT_DIR, 'pems_d3_meta_corrected.csv')
pems_export.to_csv(export_file, index=False)
print(f"已保存: {export_file}")
print(f"站点数: {len(pems_export)}")

已保存: ./output/d3_interpolation_correction/pems_d3_meta_corrected.csv
站点数: 1192


## 5. 可视化（含里程桩）

In [13]:
def create_correction_map_with_postmiles(pems_corrected, caltrans_df, fwy, direction, output_path):
    """
    创建含里程桩的修正对比地图
    
    显示:
    - Caltrans 里程桩（蓝色小点）
    - 原始 PeMS 坐标（红色）
    - 修正后坐标（绿色）
    - 修正连线
    """
    # 筛选数据
    pems_subset = pems_corrected[
        (pems_corrected['Fwy'] == str(fwy)) &
        (pems_corrected['Dir'] == direction) &
        (pems_corrected['Status'] == 'corrected')
    ]
    
    if len(pems_subset) == 0:
        print(f"无 {fwy}{direction} 数据")
        return None
    
    # 筛选对应方向的 Caltrans 数据
    if direction in ['N', 'E']:
        align_codes = ['Right', 'Right Side']
    else:
        align_codes = ['Left', 'Left Side']
    
    caltrans_subset = caltrans_df[
        (caltrans_df['Route'] == fwy) &
        (caltrans_df['AlignCode'].isin(align_codes))
    ].sort_values('Odometer')
    
    # 计算中心
    center_lat = pems_subset['Corrected_Lat'].mean()
    center_lon = pems_subset['Corrected_Lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles=None)
    
    # 底图
    folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        attr='Google', name='Google 街道'
    ).add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google', name='Google 混合'
    ).add_to(m)
    
    # ========== Caltrans 里程桩 ==========
    postmile_group = folium.FeatureGroup(name='Caltrans 里程桩')
    for _, row in caltrans_subset.iterrows():
        # 整数里程用大点
        is_integer_pm = (row['Odometer'] % 1) < 0.05 or (row['Odometer'] % 1) > 0.95
        radius = 5 if is_integer_pm else 2
        color = '#1565C0' if is_integer_pm else '#64B5F6'
        
        folium.CircleMarker(
            [row['Latitude'], row['Longitude']],
            radius=radius,
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.8,
            weight=1,
            popup=f"PM: {row['Odometer']:.2f}",
            tooltip=f"{row['Odometer']:.1f}"
        ).add_to(postmile_group)
    postmile_group.add_to(m)
    
    # 站点类型颜色
    type_colors = {
        'ML': '#E53935',
        'HV': '#8E24AA',
        'FF': '#FFC107',
    }
    
    # ========== 原始 PeMS 坐标 ==========
    original_group = folium.FeatureGroup(name='原始 PeMS 坐标')
    for _, row in pems_subset.iterrows():
        color = type_colors.get(row['Type'], '#888')
        folium.CircleMarker(
            [row['Original_Lat'], row['Original_Lon']],
            radius=8,
            color='#333',
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=f"<b>原始</b><br>{row['ID']}<br>{row['Type']}<br>PM={row['Abs_PM']:.2f}",
            tooltip=f"原始 {row['ID']}"
        ).add_to(original_group)
    original_group.add_to(m)
    
    # ========== 修正后坐标 ==========
    corrected_group = folium.FeatureGroup(name='修正后坐标')
    for _, row in pems_subset.iterrows():
        color = type_colors.get(row['Type'], '#888')
        folium.CircleMarker(
            [row['Corrected_Lat'], row['Corrected_Lon']],
            radius=8,
            color='#4CAF50',
            weight=3,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=f"<b>修正后</b><br>{row['ID']}<br>偏移={row['Correction_Dist_ft']:.0f}ft<br>插值: {row['Interp_Type']}",
            tooltip=f"修正 {row['ID']} ({row['Correction_Dist_ft']:.0f}ft)"
        ).add_to(corrected_group)
    corrected_group.add_to(m)
    
    # ========== 修正连线 ==========
    lines_group = folium.FeatureGroup(name='修正连线')
    for _, row in pems_subset.iterrows():
        if row['Correction_Dist_ft'] < 10:
            continue
        
        if row['Correction_Dist_ft'] < 100:
            color = '#FFC107'
        elif row['Correction_Dist_ft'] < 500:
            color = '#FF9800'
        else:
            color = '#F44336'
        
        folium.PolyLine(
            [[row['Original_Lat'], row['Original_Lon']],
             [row['Corrected_Lat'], row['Corrected_Lon']]],
            color=color,
            weight=2,
            dash_array='5,5',
            opacity=0.8
        ).add_to(lines_group)
    lines_group.add_to(m)
    
    # 图例
    legend = f"""
    <div style="position:fixed; bottom:50px; left:50px; z-index:1000;
                background:white; padding:12px; border:2px solid #333; border-radius:5px;
                font-size:12px;">
        <div style="font-weight:bold; margin-bottom:8px;">Fwy {fwy}{direction} 坐标修正</div>
        
        <div style="margin-top:5px;"><b>Caltrans 里程桩:</b></div>
        <div><span style="color:#1565C0;">●</span> 整数里程</div>
        <div><span style="color:#64B5F6;">·</span> 0.1 英里间隔</div>
        
        <div style="margin-top:5px;"><b>PeMS 站点:</b></div>
        <div><span style="color:#E53935;">●</span> ML 主线</div>
        <div><span style="color:#8E24AA;">●</span> HV HOV</div>
        <div><span style="color:#FFC107;">●</span> FF 连接器</div>
        
        <div style="margin-top:5px;"><b>边框:</b></div>
        <div>黑边 = 原始位置</div>
        <div>绿边 = 修正后位置</div>
        
        <div style="margin-top:5px;"><b>连线:</b></div>
        <div><span style="color:#FFC107;">—</span> <100ft</div>
        <div><span style="color:#FF9800;">—</span> 100-500ft</div>
        <div><span style="color:#F44336;">—</span> >500ft</div>
        
        <div style="margin-top:5px; font-size:10px;">站点数: {len(pems_subset)}</div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend))
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"已保存: {output_path}")
    return m


print("可视化函数定义完成")

可视化函数定义完成


In [14]:
# 为主要高速生成地图
main_highways = [99, 80, 5, 50]

for fwy in main_highways:
    for direction in ['N', 'S', 'E', 'W']:
        # 检查是否有数据
        count = len(pems_corrected[
            (pems_corrected['Fwy'] == str(fwy)) &
            (pems_corrected['Dir'] == direction) &
            (pems_corrected['Status'] == 'corrected')
        ])
        
        if count > 0:
            create_correction_map_with_postmiles(
                pems_corrected,
                caltrans_df,
                fwy,
                direction,
                os.path.join(OUTPUT_DIR, f'correction_{fwy}{direction}_postmiles.html')
            )

已保存: ./output/d3_interpolation_correction/correction_99N_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_99S_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_80E_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_80W_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_5N_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_5S_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_50E_postmiles.html
已保存: ./output/d3_interpolation_correction/correction_50W_postmiles.html


## 6. 全局概览地图

In [15]:
def create_overview_map(pems_corrected, output_path):
    """创建 D3 全局概览地图"""
    corrected = pems_corrected[pems_corrected['Status'] == 'corrected']
    
    center_lat = corrected['Corrected_Lat'].mean()
    center_lon = corrected['Corrected_Lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles=None)
    
    folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google', name='Google 混合'
    ).add_to(m)
    
    type_colors = {
        'ML': '#E53935',
        'HV': '#8E24AA',
        'FF': '#FFC107',
    }
    
    # 原始位置
    original_group = folium.FeatureGroup(name='原始位置')
    for _, row in corrected.iterrows():
        color = type_colors.get(row['Type'], '#888')
        folium.CircleMarker(
            [row['Original_Lat'], row['Original_Lon']],
            radius=4,
            color=color,
            fill=True,
            fillOpacity=0.7,
            weight=1
        ).add_to(original_group)
    original_group.add_to(m)
    
    # 修正后位置
    corrected_group = folium.FeatureGroup(name='修正后位置')
    for _, row in corrected.iterrows():
        color = type_colors.get(row['Type'], '#888')
        folium.CircleMarker(
            [row['Corrected_Lat'], row['Corrected_Lon']],
            radius=4,
            color='#4CAF50',
            fill=True,
            fillColor=color,
            fillOpacity=0.8,
            weight=2
        ).add_to(corrected_group)
    corrected_group.add_to(m)
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"已保存: {output_path}")
    return m


# 生成全局地图
overview_map = create_overview_map(
    pems_corrected,
    os.path.join(OUTPUT_DIR, 'correction_d3_overview.html')
)
overview_map

已保存: ./output/d3_interpolation_correction/correction_d3_overview.html


## 总结

### 插值方法

```
PM1 (10.0) -------- Abs_PM (10.3) -------- PM2 (10.4)
   |                    |                    |
(lat1,lon1)    线性插值计算          (lat2,lon2)
                    ↓
              (lat_new, lon_new)
              
t = (10.3 - 10.0) / (10.4 - 10.0) = 0.75
lat_new = lat1 + 0.75 * (lat2 - lat1)
lon_new = lon1 + 0.75 * (lon2 - lon1)
```

### 输出文件

| 文件 | 说明 |
|------|------|
| `pems_d3_interpolated.csv` | 完整结果 |
| `pems_d3_meta_corrected.csv` | 修正后元数据 |
| `correction_{Fwy}{Dir}_postmiles.html` | 含里程桩的详细地图 |
| `correction_d3_overview.html` | 全局概览 |